# 015 — Multi-Agent Coordination with LangGraph
**Agentic Orchestration** | Supervisor routes to domain specialists

Architecture:
```
User Query
    ↓
Supervisor Agent  ← decides which specialist to call
    ├── ML Expert Agent     (machine learning questions)
    ├── NLP Expert Agent    (language processing questions)
    └── General Agent       (everything else)
         ↓
Synthesizer → Final Answer
```


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build per-domain RAG retrievers ──────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)

def build_store(texts, label):
    docs = [Document(page_content=c, metadata={"domain": label})
            for text in texts for c in splitter.split_text(text[:12000])]
    store = FAISS.from_documents(docs, embeddings)
    return store.as_retriever(search_kwargs={"k": 3})

ml_retriever  = build_store([ml_text, dl_text], "ml")
nlp_retriever = build_store([nlp_text],          "nlp")
gen_retriever  = build_store([ai_text, rag_text], "general")

print("✓ Domain retrievers: ML, NLP, General")


In [ ]:
# ── Specialist agent factory ─────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def specialist_agent(retriever, domain: str):
    prompt = ChatPromptTemplate.from_template(
        f"You are an expert in {domain}. Answer based on the context provided.\n\n"
        "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    def run(question: str) -> str:
        docs = retriever.invoke(question)
        ctx  = "\n\n".join(d.page_content for d in docs)
        return (prompt | llm | StrOutputParser()).invoke(
            {"context": ctx[:3000], "question": question}
        )
    return run

ml_agent  = specialist_agent(ml_retriever,  "machine learning and deep learning")
nlp_agent = specialist_agent(nlp_retriever, "natural language processing")
gen_agent = specialist_agent(gen_retriever, "artificial intelligence and RAG")
print("✓ ML, NLP, and General agents ready")


In [ ]:
# ── Supervisor routing ───────────────────────────────────────────────────
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Route the question to the best specialist.\n"
        "Reply with exactly one word: 'ml', 'nlp', or 'general'."
    )),
    ("human", "Question: {question}"),
])
supervisor_chain = supervisor_prompt | llm | StrOutputParser()

test_routes = [
    "How does gradient descent work?",
    "What is tokenization in NLP?",
    "What is retrieval-augmented generation?",
    "How does BERT handle masked language modeling?",
]
for q in test_routes:
    route = supervisor_chain.invoke({"question": q}).strip().lower()
    print(f"  [{route:8}] {q}")


In [ ]:
# ── LangGraph multi-agent graph ──────────────────────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, END

class MultiAgentState(TypedDict):
    question: str
    route:    str
    draft:    str
    answer:   str

def supervisor_node(state: MultiAgentState):
    route = supervisor_chain.invoke({"question": state["question"]}).strip().lower()
    if route not in ("ml", "nlp", "general"):
        route = "general"
    print(f"  Supervisor → {route}")
    return {**state, "route": route}

def ml_node(state: MultiAgentState):
    return {**state, "draft": ml_agent(state["question"])}

def nlp_node(state: MultiAgentState):
    return {**state, "draft": nlp_agent(state["question"])}

def general_node(state: MultiAgentState):
    return {**state, "draft": gen_agent(state["question"])}

def synthesize(state: MultiAgentState):
    synth_prompt = ChatPromptTemplate.from_template(
        "Refine this answer to be clear and well-structured:\n\n{draft}\n\nFinal answer:"
    )
    final = (synth_prompt | llm_smart | StrOutputParser()).invoke({"draft": state["draft"]})
    return {**state, "answer": final}

def route_to_specialist(state: MultiAgentState):
    return state["route"]

graph = StateGraph(MultiAgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("ml",         ml_node)
graph.add_node("nlp",        nlp_node)
graph.add_node("general",    general_node)
graph.add_node("synthesize", synthesize)
graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_to_specialist,
                             {"ml": "ml", "nlp": "nlp", "general": "general"})
for specialist in ("ml", "nlp", "general"):
    graph.add_edge(specialist, "synthesize")
graph.add_edge("synthesize", END)
multi_agent_app = graph.compile()
print("✓ Multi-agent graph compiled")


In [ ]:
# ── Run multi-agent pipeline ─────────────────────────────────────────────
for q in test_routes[:2]:
    print(f"\nQ: {q}")
    result = multi_agent_app.invoke(
        {"question": q, "route": "", "draft": "", "answer": ""}
    )
    print(f"Route  : {result['route']}")
    print(f"Answer : {result['answer'][:350]}")


## Key Takeaways
- Supervisor architecture scales: add specialists without touching core routing logic
- Each specialist has its own retriever + prompt — clean separation of concerns
- The synthesizer step ensures consistent output format regardless of which specialist ran
- For production: add memory sharing between agents (episodic store)
